In [1]:
!pip install --upgrade lightgbm -q
!pip install --upgrade category-encoders
!pip install joblib

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.5/3.5 MB 49.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 107.9/107.9 kB 3.7 MB/s eta 0:00:00


In [2]:
# 1. Системні утиліти, файли та Google Диск
import os
import json
import numpy as np
import pandas as pd
from google.colab import drive

# 2. Scikit-learn: побудова конвеєрів та трансформації
from sklearn import set_config
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import FunctionTransformer
from sklearn.base import clone

# 3. Валідація, оцінка якості та калібрування
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_auc_score
from sklearn.isotonic import IsotonicRegression

# 4. Моделі градієнтного бустингу
import lightgbm as lgb
import xgboost as xgb

# 5. Математична оптимізація (для підбору ваг блендингу замість фіксованих 0.5/0.5)
from scipy.optimize import minimize

from category_encoders import TargetEncoder

from sklearn.linear_model import LogisticRegression

import shap
import matplotlib.pyplot as plt

import joblib

# Монтування Диска та відображення у вигляді DataFrame
drive.mount('/content/drive', force_remount=True)
set_config(transform_output="pandas")

Mounted at /content/drive


In [3]:
PROJECT_PATH = '/content/drive/MyDrive/Colab Notebooks/Home_Credit_Project'

X_train_full = pd.read_parquet(f'{PROJECT_PATH}/X_train_full.parquet')
y_train = pd.read_parquet(f'{PROJECT_PATH}/y_train.parquet')['TARGET']
X_test_full = pd.read_parquet(f'{PROJECT_PATH}/X_test_full.parquet')
test_id = pd.read_parquet(f'{PROJECT_PATH}/test_id.parquet')['SK_ID_CURR']

In [4]:
%%writefile "/content/drive/MyDrive/Colab Notebooks/Home_Credit_Project/Production/predict.py"
import os
import sys
import joblib
import numpy as np
import pandas as pd

from pipeline import (
    fillna_col,
    to_category,
    fillna_category,
    car_age,
    fix_day_employed,
    custom_feature_full,
)

sys.modules["__main__"].fillna_col = fillna_col
sys.modules["__main__"].to_category = to_category
sys.modules["__main__"].fillna_category = fillna_category
sys.modules["__main__"].car_age = car_age
sys.modules["__main__"].fix_day_employed = fix_day_employed
sys.modules["__main__"].custom_feature_full = custom_feature_full

class CreditScoringService:
  def __init__(self, model_dir: str):
    self.lgbm_folds = joblib.load(os.path.join(model_dir, "lgbm_folds.joblib"))
    self.xgb_folds = joblib.load(os.path.join(model_dir, "xgb_folds.joblib"))
    self.meta_model = joblib.load(os.path.join(model_dir, "meta_model.joblib"))

  def assign_risk_grade(self, probabilities: np.ndarray) -> np.ndarray:
    conditions = [
        probabilities <= 0.05,
        (probabilities > 0.05) & (probabilities <= 0.15),
        (probabilities > 0.15) & (probabilities <= 0.30)
    ]
    choices = ['A_Low_Risk', 'B_Medium_Risk', 'C_High_Risk']
    return np.select(conditions, choices, default = 'D_Reject')
  def predict(self, df: pd.DataFrame) -> pd.DataFrame:
    lgbm_preds = np.mean(
        [model.predict_proba(df)[:, 1] for model in self.lgbm_folds], axis = 0
    )
    xgb_preds = np.mean(
        [model.predict_proba(df)[:, 1] for model in self.xgb_folds], axis = 0
    )
    X_meta = np.column_stack([lgbm_preds, xgb_preds])
    final_probs = self.meta_model.predict_proba(X_meta)[:, 1]
    risk_grades = self.assign_risk_grade(final_probs)

    result = pd.DataFrame({
            "probability": np.round(final_probs, 4),
            "risk_grade": risk_grades,
            "lgbm_prob": np.round(lgbm_preds, 4),
            "xgb_prob": np.round(xgb_preds, 4),
        })
    if "SK_ID_CURR" in df.columns:
      result.insert(0, "SK_ID_CURR", df["SK_ID_CURR"].values)
    return result

Writing /content/drive/MyDrive/Colab Notebooks/Home_Credit_Project/Production/predict.py


In [8]:
import os
import sys
import importlib
import pandas as pd

MODEL_PATH = '/content/drive/MyDrive/Colab Notebooks/Home_Credit_Project/Models'
PROD_PATH = '/content/drive/MyDrive/Colab Notebooks/Home_Credit_Project/Production'

sample_csv = os.path.join(PROD_PATH, 'sample_clients.csv')
X_test_full.iloc[:5].to_csv(sample_csv, index=False)

if PROD_PATH not in sys.path:
    sys.path.append(PROD_PATH)

import predict
importlib.reload(predict)
from predict import CreditScoringService

service = CreditScoringService(MODEL_PATH)
sample_df = pd.read_csv(sample_csv)

predictions = service.predict(sample_df)
display(predictions)

,probability,risk_grade,lgbm_prob,xgb_prob
0,0.0507,B_Medium_Risk,0.0583,0.0507
1,0.1147,B_Medium_Risk,0.1624,0.1551
2,0.0460,A_Low_Risk,0.0422,0.0425
3,0.0464,A_Low_Risk,0.0464,0.0403
4,0.1583,C_High_Risk,0.2157,0.1896


In [ ]:
skf = StratifiedKFold(n_splits = 5, shuffle=True, random_state=42)

In [ ]:
pcb_cols = [ col for col in X_train_full if col.startswith("PREV_")]
bereua_cols = [ col for col in X_train_full if col.startswith("BUREAU_") and not col.startswith("BUREAU_BB_")]
custom_fill_na_cols = pcb_cols + bereua_cols

In [ ]:
cat_cols = set(X_train_full.select_dtypes(include = ['string', 'object', 'category']).columns)
cat_cols_list = list(cat_cols)

In [ ]:
def fillna_col (df):
    return df.fillna(0)
num_zero_transform = FunctionTransformer(fillna_col)
def to_category(df):
    return df.astype("category")
def fillna_category(df):
    return df.fillna("None")
cat_with_na = Pipeline([
    ("fill_na", FunctionTransformer(fillna_category)),
    ("to_category", FunctionTransformer(to_category))
])
def car_age(df):
    return df.fillna(-1)
fill_car_age = FunctionTransformer(car_age)
def fix_day_employed(df):
    df = df.copy()
    if "DAYS_EMPLOYED" in df.columns:
        df["DE_ANOM"] = (df["DAYS_EMPLOYED"] == 365243).astype(int)
        df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace({365243: np.nan})
    return df
fix_de = FunctionTransformer(fix_day_employed)

def custom_feature_full(df):
    df = df.copy()

    monthly_income = df["AMT_INCOME_TOTAL"] / 12
    df["MONTHLY_INCOME_TO_PREV_CREDIT"] = monthly_income / (df["PREV_AMT_CREDIT_MEAN"] + 1)
    df["PREV_ANNUITY_TO_MONTHLY_INCOME"] = df["PREV_AMT_ANNUITY_MEAN"] / (monthly_income + 1)

    fix_member = df["CNT_FAM_MEMBERS"].fillna(1).replace(0, 1)
    df["INCOME_PER_FAMILY_MEMBER"] = df["AMT_INCOME_TOTAL"] / fix_member

    ext_cols = ["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]
    df["EXT_SOURCE_MIN"] = df[ext_cols].min(axis=1)
    df["EXT_SOURCE_MEAN"] = df[ext_cols].mean(axis=1)
    df["EXT_SOURCE_MAX"] = df[ext_cols].max(axis=1)
    df["EXT_SOURCE_STD"] = df[ext_cols].std(axis=1)

    e1 = df["EXT_SOURCE_1"].fillna(df["EXT_SOURCE_1"].median()).fillna(0.5)
    e2 = df["EXT_SOURCE_2"].fillna(df["EXT_SOURCE_2"].median()).fillna(0.5)
    e3 = df["EXT_SOURCE_3"].fillna(df["EXT_SOURCE_3"].median()).fillna(0.5)
    df["EXT_SOURCE_PROD"] = e1 * e2 * e3
    df["EXT_SOURCE_COUNT"] = df[ext_cols].notnull().sum(axis=1)

    df["PREV_DAYS_DECISION_SPAN"] = df["PREV_DAYS_DECISION_MAX"] - df["PREV_DAYS_DECISION_MIN"]

    df["BUREAU_DAYS_CREDIT_SPAN"] = df["BUREAU_DAYS_CREDIT_MAX"] - df["BUREAU_DAYS_CREDIT_MIN"]

    return df
custom_feature_step = FunctionTransformer(custom_feature_full)

Overwriting /content/drive/MyDrive/Colab Notebooks/Home_Credit_Project/Production/pipeline.py


In [ ]:
te_cols = ['OCCUPATION_TYPE', 'ORGANIZATION_TYPE']
target_encoder_step = TargetEncoder(
    smoothing = 10.0
)

In [ ]:
preprocessor = ColumnTransformer (
    transformers = [
        ("tar_encd_col", target_encoder_step, te_cols),
        ("cat_col", cat_with_na, cat_cols_list),
        ("custom_col", num_zero_transform, custom_fill_na_cols),
    ],
    remainder="passthrough",)

In [ ]:

model_lgbm = Pipeline([
    ("custom_feature", custom_feature_step),
    ("de_fix", fix_de),
    ("prep", preprocessor),
    ("LGBM", lgb.LGBMClassifier(
        random_state=17,
        n_jobs=4,
        verbose=-1
    ))
])

model_xgb = Pipeline([
    ("custom_feature", custom_feature_step),
    ("de_fix", fix_de),
    ("prep", preprocessor),
    ("XGBoost", xgb.XGBClassifier(
        tree_method="hist",
        device="cuda",
        random_state=17,
        verbosity=0,
        enable_categorical=True,
        eval_metric="auc"
    ))
])

In [ ]:
PARAMS_PATH = f'{PROJECT_PATH}/best_params.json'
with open(PARAMS_PATH, 'r') as f:
    raw_data = json.load(f)

print("Знайдені ключі у файлі:", list(raw_data.keys()))
print("Повний вміст:", raw_data)

Знайдені ключі у файлі: ['params', 'scores']
Повний вміст: {'params': {'lgbm': {'num_leaves': 64, 'min_child_samples': 115, 'learning_rate': 0.03178419464238957, 'n_estimators': 495, 'reg_alpha': 9.512942358246606, 'reg_lambda': 0.08276556896960584}, 'xgb': {'max_depth': 9, 'learning_rate': 0.03782974344777758, 'n_estimators': 365, 'min_child_weight': 39, 'subsample': 0.864770026172548, 'colsample_bytree': 0.6994626170912169, 'reg_alpha': 5.268337992217248, 'reg_lambda': 3.6619923811945134}}, 'scores': {'lgbm': 0.7805193416139804, 'xgb': 0.7823266643770307}}


In [ ]:
# Шлях до файлу конфігурацій
PARAMS_PATH = f'{PROJECT_PATH}/best_params.json'

# Завантаження збережених параметрів
with open(PARAMS_PATH, 'r') as f:
    best_params = json.load(f)

# Дістаємо параметри через проміжний ключ 'params'
best_params_lgbm = best_params['params']['lgbm']
best_params_xgb = best_params['params']['xgb']

In [ ]:
oof_result_lgbm = np.zeros(len(X_train_full))
test_result_lgbm = np.zeros(len(X_test_full))
model_list_lgbm = []
for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_full, y_train), 1):
    X_tr, y_tr = X_train_full.iloc[train_idx], y_train.iloc[train_idx]
    X_val, y_val = X_train_full.iloc[val_idx], y_train.iloc[val_idx]
    fold_model = clone(model_lgbm)
    fold_model.set_params(**{f"LGBM__{k}": v for k, v in best_params_lgbm.items()})
    fold_model.fit(X_tr, y_tr)
    val_preds = fold_model.predict_proba(X_val)[:, 1]
    oof_result_lgbm[val_idx] = val_preds

    test_result_lgbm += fold_model.predict_proba(X_test_full)[:, 1] / skf.n_splits
    fold_auc = roc_auc_score(y_val, val_preds)
    print(f"Fold {fold} ROC-AUC: {fold_auc:.5f}")

    model_list_lgbm.append(fold_model)
overall_roc_auc = roc_auc_score(y_train, oof_result_lgbm)
print(f"Загальний ROC-AUC: {overall_roc_auc:.5f}")

Fold 1 ROC-AUC: 0.77795
Fold 2 ROC-AUC: 0.78935
Fold 3 ROC-AUC: 0.78084
Fold 4 ROC-AUC: 0.78502
Fold 5 ROC-AUC: 0.77764
Загальний ROC-AUC: 0.78212


In [ ]:
oof_result_xgb = np.zeros(len(X_train_full))
test_result_xgb = np.zeros(len(X_test_full))
model_xgb_list = []
for fold, (train_idx, val_idx) in enumerate(skf.split(X_train_full, y_train), 1):
    X_tr, y_tr = X_train_full.iloc[train_idx], y_train.iloc[train_idx]
    X_val, y_val = X_train_full.iloc[val_idx], y_train.iloc[val_idx]
    fold_model = clone(model_xgb)
    fold_model.set_params(**{f"XGBoost__{k}": v for k, v in best_params_xgb.items()})
    fold_model.fit(X_tr, y_tr)
    oof_pred = fold_model.predict_proba(X_val)[:, 1]
    oof_result_xgb[val_idx] = oof_pred
    fold_auc = roc_auc_score(y_val, oof_pred)
    print(f"номер фолда {fold}, roc_auc: {fold_auc:.5f}")
    test_res = fold_model.predict_proba(X_test_full)[:, 1]
    test_result_xgb += test_res / skf.n_splits
    model_xgb_list.append(fold_model)
model_xgb_auc = roc_auc_score(y_train, oof_result_xgb)
print(f"roc_auc моделей: {model_xgb_auc:.5f}")

номер фолда 1, roc_auc: 0.78036
номер фолда 2, roc_auc: 0.79102
номер фолда 3, roc_auc: 0.78238
номер фолда 4, roc_auc: 0.78600
номер фолда 5, roc_auc: 0.77915
roc_auc моделей: 0.78375


In [ ]:
calibrator_lgbm = IsotonicRegression(out_of_bounds = 'clip', y_min = 0.0, y_max = 1.0)
calibrator_xgb = IsotonicRegression(out_of_bounds = 'clip', y_min = 0.0, y_max = 1.0)
calibrator_lgbm.fit(oof_result_lgbm, y_train)
calibrator_xgb.fit(oof_result_xgb, y_train)
oof_result_lgbm_calib = calibrator_lgbm.predict(oof_result_lgbm)
oof_result_xgb_calib = calibrator_xgb.predict(oof_result_xgb)
oof_blend_calib = 0.5 * oof_result_xgb_calib + 0.5 * oof_result_lgbm_calib
oof_blend_res = roc_auc_score(y_train, oof_blend_calib)
print(f"roc_auc загальних результатів: {oof_blend_res:.5f}")

roc_auc загальних результатів: 0.78505


In [ ]:
test_result_lgbm_calib = calibrator_lgbm.predict(test_result_lgbm)
test_result_xgb_calib = calibrator_xgb.predict(test_result_xgb)

In [ ]:
X_meta_train = np.column_stack([oof_result_lgbm_calib , oof_result_xgb_calib ])
X_meta_test = np.column_stack([test_result_xgb_calib, test_result_lgbm_calib])

In [ ]:
meta_model = LogisticRegression(C=1.0, solver='lbfgs', random_state=17)
meta_model.fit(X_meta_train, y_train)
meta_oof_predict = meta_model.predict_proba(X_meta_train)[:, 1]
meta_test_predict = meta_model.predict_proba(X_meta_test)[:, 1]

In [ ]:
probabilities = meta_test_predict.copy()
conditions = [
    probabilities < 0.05,
    (probabilities >= 0.05) & (probabilities <= 0.15),
    (probabilities > 0.15) & (probabilities <= 0.30)
]
choices = ['A_Low_Risk', 'B_Medium_Risk', 'C_High_Risk']
risk_grade = np.select(conditions, choices, default = 'D_Reject')

In [ ]:
MODEL_PATH = '/content/drive/MyDrive/Colab Notebooks/Home_Credit_Project/Models'

In [ ]:
meta_model_path = os.path.join(MODEL_PATH, 'meta_model.joblib')
joblib.dump(meta_model, meta_model_path)
lgbm_path = os.path.join(MODEL_PATH, 'lgbm_folds.joblib')
xgb_path = os.path.join(MODEL_PATH, 'xgb_folds.joblib')

joblib.dump(model_list_lgbm, lgbm_path, compress=3)
joblib.dump(model_xgb_list, xgb_path, compress=3)

print("Усі моделі успішно збережено на ваш Google Диск!")

Усі моделі успішно збережено на ваш Google Диск!


In [ ]:
import os

print("1. Перевірка змінних у пам'яті:")
print(f"Кількість моделей у model_list_lgbm: {len(model_list_lgbm)}")
print(f"Кількість моделей у model_xgb_list: {len(model_xgb_list)}")
print(f"Об'єкт meta_model: {type(meta_model)}")
print("-" * 50)

print("2. Файли фізично у папці MODEL_PATH та їхній розмір:")
for filename in os.listdir(MODEL_PATH):
    filepath = os.path.join(MODEL_PATH, filename)
    size_kb = os.path.getsize(filepath) / 1024
    print(f" - {filename}: {size_kb:.2f} KB")

1. Перевірка змінних у пам'яті:
Кількість моделей у model_list_lgbm: 5
Кількість моделей у model_xgb_list: 5
Об'єкт meta_model: <class 'sklearn.linear_model._logistic.LogisticRegression'>
--------------------------------------------------
2. Файли фізично у папці MODEL_PATH та їхній розмір:
 - meta_model.joblib: 0.86 KB
 - lgbm_folds.joblib: 7514.06 KB
 - xgb_folds.joblib: 6772.05 KB


In [ ]:
submission = pd.DataFrame({
    'SK_ID_CURR': test_id,
    'TARGET': meta_test_predict,
    'RISK_GRADE': risk_grade
})

output_path = os.path.join(PROJECT_PATH, 'final_credit_decisions.csv')

submission.to_csv(output_path, index=False)
print("Файл сабмішну успішно збережено на Google Диск:", output_path)

Файл сабмішну успішно збережено на Google Диск: /content/drive/MyDrive/Colab Notebooks/Home_Credit_Project/final_credit_decisions.csv
